# Toy Brick Data Preparation - Snowflake Edition

This notebook prepares toy brick data from Rebrickable for optimization analysis using Snowflake.

## Prerequisites
- Snowflake account with appropriate database/schema permissions
- Rebrickable CSV files downloaded from https://rebrickable.com/downloads/
- Files to download:
  - sets.csv.gz
  - inventory_sets.csv.gz
  - inventories.csv.gz
  - inventory_parts.csv.gz
  - parts.csv.gz
  - colors.csv.gz
  - themes.csv.gz

## Configuration

In [ ]:
import snowflake.snowpark as snowpark
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, lit, when, count, sum as sum_
from snowflake.snowpark.types import *
import pandas as pd

# Configuration - update these for your environment
DATABASE = "TOY_BRICK_DB"
SCHEMA = "RAW_DATA"
STAGE_NAME = "REBRICKABLE_STAGE"

# Optional: If running in Snowflake notebook, session is provided
# Otherwise, create session with your credentials
try:
    session = snowpark.Session.builder.getOrCreate()
except:
    # Create session manually if not in Snowflake environment
    connection_parameters = {
        "account": "your_account",
        "user": "your_user",
        "password": "your_password",
        "role": "your_role",
        "warehouse": "your_warehouse",
        "database": DATABASE,
        "schema": SCHEMA
    }
    session = Session.builder.configs(connection_parameters).create()

print(f"Snowflake version: {session.sql('SELECT CURRENT_VERSION()').collect()[0][0]}")
print(f"Current database: {session.get_current_database()}")
print(f"Current schema: {session.get_current_schema()}")

## Step 1: Create Database and Schema

In [ ]:
# ============================================================================
# DATABASE AND SCHEMA ALREADY EXIST - Just switch to use them
# ============================================================================

# Create database and schema if they don't exist (already done)
# session.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE}").collect()
# session.sql(f"CREATE SCHEMA IF NOT EXISTS {DATABASE}.{SCHEMA}").collect()

session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()

print(f"Using database and schema: {DATABASE}.{SCHEMA}")

## Step 2: Create Stage for Loading Data

We'll create an internal stage to upload the Rebrickable CSV files.

In [ ]:
# ============================================================================
# STAGE ALREADY CREATED - The following is commented out
# Data was uploaded using: snow stage copy <file>.csv @TOY_BRICK_DB.RAW_DATA.REBRICKABLE_STAGE -c demo
# ============================================================================

# # Create internal stage
# session.sql(f"""
# CREATE STAGE IF NOT EXISTS {STAGE_NAME}
# FILE_FORMAT = (TYPE = CSV 
#                FIELD_DELIMITER = ',' 
#                SKIP_HEADER = 1 
#                FIELD_OPTIONALLY_ENCLOSED_BY = '"'
#                NULL_IF = ('NULL', 'null', ''))
# """).collect()

print(f"Stage already exists: {STAGE_NAME}")
print("\nData was uploaded using SnowCLI commands like:")
print(f"snow stage copy themes.csv @TOY_BRICK_DB.RAW_DATA.{STAGE_NAME} -c demo")
print(f"snow stage copy sets.csv @TOY_BRICK_DB.RAW_DATA.{STAGE_NAME} -c demo")
print("... etc for all CSV files")

## Step 3: Create Raw Tables

In [ ]:
# ============================================================================
# TABLES ALREADY CREATED - The following commands are commented out
# Tables were created with correct schemas matching the CSV files
# ============================================================================

# # Create themes table
# session.sql("""
# CREATE OR REPLACE TABLE themes (
#     id INT PRIMARY KEY,
#     name VARCHAR(255),
#     parent_id INT
# )
# """).collect()

# # Create sets table (includes img_url from CSV)
# session.sql("""
# CREATE OR REPLACE TABLE sets (
#     set_num VARCHAR(20) PRIMARY KEY,
#     name VARCHAR(500),
#     year INT,
#     theme_id INT,
#     num_parts INT,
#     img_url VARCHAR(500)
# )
# """).collect()

# # Create inventories table
# session.sql("""
# CREATE OR REPLACE TABLE inventories (
#     id INT PRIMARY KEY,
#     version INT,
#     set_num VARCHAR(20)
# )
# """).collect()

# # Create colors table (includes additional columns from CSV)
# session.sql("""
# CREATE OR REPLACE TABLE colors (
#     id INT PRIMARY KEY,
#     name VARCHAR(100),
#     rgb VARCHAR(6),
#     is_trans VARCHAR(10),  -- CSV has True/False as text
#     num_parts INT,
#     num_sets INT,
#     y1 INT,
#     y2 INT
# )
# """).collect()

# # Create parts table
# session.sql("""
# CREATE OR REPLACE TABLE parts (
#     part_num VARCHAR(50) PRIMARY KEY,
#     name VARCHAR(500),
#     part_cat_id INT,
#     part_material VARCHAR(50)
# )
# """).collect()

# # Create inventory_parts table (includes img_url from CSV)
# session.sql("""
# CREATE OR REPLACE TABLE inventory_parts (
#     inventory_id INT,
#     part_num VARCHAR(50),
#     color_id INT,
#     quantity INT,
#     is_spare VARCHAR(10),  -- CSV has True/False as text
#     img_url VARCHAR(500)
# )
# """).collect()

# # Create inventory_sets table  
# session.sql("""
# CREATE OR REPLACE TABLE inventory_sets (
#     inventory_id INT,
#     set_num VARCHAR(20),
#     quantity INT
# )
# """).collect()

print("Tables already created - skipping CREATE TABLE commands")

## Step 4: Load Data from Stage

After uploading the CSV files to the stage, run these commands to load the data.

In [ ]:
# ============================================================================
# DATA ALREADY LOADED - The following commands are commented out
# Data was loaded via SnowCLI from local CSV files (not gzipped)
# ============================================================================

# # Load themes
# session.sql(f"""
# COPY INTO themes
# FROM @{STAGE_NAME}/themes.csv
# FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"' NULL_IF = ('', 'NULL', 'null'))
# ON_ERROR = CONTINUE
# """).collect()

# # Load sets
# session.sql(f"""
# COPY INTO sets
# FROM @{STAGE_NAME}/sets.csv
# FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"' NULL_IF = ('', 'NULL', 'null'))
# ON_ERROR = CONTINUE
# """).collect()

# # Load inventories
# session.sql(f"""
# COPY INTO inventories
# FROM @{STAGE_NAME}/inventories.csv
# FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"' NULL_IF = ('', 'NULL', 'null'))
# ON_ERROR = CONTINUE
# """).collect()

# # Load colors
# session.sql(f"""
# COPY INTO colors
# FROM @{STAGE_NAME}/colors.csv
# FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"' NULL_IF = ('', 'NULL', 'null'))
# ON_ERROR = CONTINUE
# """).collect()

# # Load parts
# session.sql(f"""
# COPY INTO parts
# FROM @{STAGE_NAME}/parts.csv
# FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"' NULL_IF = ('', 'NULL', 'null'))
# ON_ERROR = CONTINUE
# """).collect()

# # Load inventory_parts
# session.sql(f"""
# COPY INTO inventory_parts
# FROM @{STAGE_NAME}/inventory_parts.csv
# FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"' NULL_IF = ('', 'NULL', 'null'))
# ON_ERROR = CONTINUE
# """).collect()

# # Load inventory_sets
# session.sql(f"""
# COPY INTO inventory_sets
# FROM @{STAGE_NAME}/inventory_sets.csv
# FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"' NULL_IF = ('', 'NULL', 'null'))
# ON_ERROR = CONTINUE
# """).collect()

print("Data already loaded - skipping COPY INTO commands")

## Step 5: Verify Data Load

In [ ]:
# Check row counts for all tables
tables = [
    'themes', 'sets', 'inventories', 'colors', 'part_categories', 'parts', 
    'inventory_parts', 'inventory_sets', 'minifigs', 'inventory_minifigs',
    'elements', 'part_relationships', 'part_color_pairs', 'set_part_requirements'
]

for table in tables:
    count = session.sql(f"SELECT COUNT(*) as cnt FROM {table}").collect()[0]['CNT']
    print(f"{table}: {count:,} rows")

In [ ]:
# Sample data from sets
print("Sample sets data:")
session.sql("SELECT * FROM sets LIMIT 10").show()

## Step 6: Create Analytical Views

Create views that join the data for easier analysis.

In [ ]:
# ============================================================================
# VIEW ALREADY CREATED - The following is commented out
# Note: is_spare is VARCHAR ('True'/'False'), not BOOLEAN
# ============================================================================

# # Create comprehensive set parts view
# session.sql("""
# CREATE OR REPLACE VIEW v_set_parts AS
# SELECT 
#     s.set_num,
#     s.name as set_name,
#     s.year,
#     s.theme_id,
#     t.name as theme_name,
#     s.num_parts as total_parts,
#     ip.part_num,
#     p.name as part_name,
#     ip.color_id,
#     c.name as color_name,
#     c.rgb as color_rgb,
#     ip.quantity,
#     ip.is_spare
# FROM sets s
# JOIN themes t ON s.theme_id = t.id
# JOIN inventories inv ON s.set_num = inv.set_num
# JOIN inventory_parts ip ON inv.id = ip.inventory_id
# JOIN parts p ON ip.part_num = p.part_num
# JOIN colors c ON ip.color_id = c.id
# WHERE ip.is_spare = 'False'
# """).collect()

print("View v_set_parts already exists")

In [ ]:
# ============================================================================
# VIEW ALREADY CREATED - The following is commented out
# ============================================================================

# # Create part_color combination view for optimization
# session.sql("""
# CREATE OR REPLACE VIEW v_part_color_inventory AS
# SELECT 
#     part_num,
#     color_id,
#     part_name,
#     color_name,
#     SUM(quantity) as total_quantity
# FROM v_set_parts
# GROUP BY part_num, color_id, part_name, color_name
# """).collect()

print("View v_part_color_inventory already exists")

## Step 7: Create Optimization-Ready Tables

These tables will be used in the optimization notebooks.

In [ ]:
# ============================================================================
# TABLE ALREADY CREATED - The following is commented out
# ============================================================================

# # Create a table with part-color pairs for easier optimization
# session.sql("""
# CREATE OR REPLACE TABLE part_color_pairs AS
# SELECT DISTINCT
#     part_num || '_' || color_id as part_color_id,
#     part_num,
#     color_id,
#     part_name,
#     color_name
# FROM v_set_parts
# """).collect()

print("Table part_color_pairs already exists")

In [ ]:
# ============================================================================
# TABLE ALREADY CREATED - The following is commented out
# ============================================================================

# # Create set requirements matrix
# session.sql("""
# CREATE OR REPLACE TABLE set_part_requirements AS
# SELECT 
#     set_num,
#     set_name,
#     year,
#     theme_name,
#     part_num || '_' || color_id as part_color_id,
#     part_num,
#     color_id,
#     quantity
# FROM v_set_parts
# """).collect()

print("Table set_part_requirements already exists")

## Step 8: Generate Summary Statistics

In [ ]:
# Summary statistics
print("=" * 60)
print("TOY BRICK DATA SUMMARY")
print("=" * 60)

# Total sets
total_sets = session.sql("SELECT COUNT(DISTINCT set_num) as cnt FROM sets").collect()[0]['CNT']
print(f"Total Sets: {total_sets:,}")

# Total unique parts
total_parts = session.sql("SELECT COUNT(*) as cnt FROM parts").collect()[0]['CNT']
print(f"Total Unique Parts: {total_parts:,}")

# Total unique colors
total_colors = session.sql("SELECT COUNT(*) as cnt FROM colors").collect()[0]['CNT']
print(f"Total Unique Colors: {total_colors:,}")

# Total part-color combinations
total_combos = session.sql("SELECT COUNT(*) as cnt FROM part_color_pairs").collect()[0]['CNT']
print(f"Total Part-Color Combinations: {total_combos:,}")

# Year range
year_stats = session.sql("""
    SELECT MIN(year) as min_year, MAX(year) as max_year 
    FROM sets WHERE year IS NOT NULL
""").collect()[0]
print(f"Year Range: {year_stats['MIN_YEAR']} - {year_stats['MAX_YEAR']}")

# Top themes by set count
print("\nTop 10 Themes by Set Count:")
session.sql("""
    SELECT theme_name, COUNT(*) as set_count
    FROM v_set_parts
    GROUP BY theme_name
    ORDER BY set_count DESC
    LIMIT 10
""").show()

print("\n" + "=" * 60)
print("Data preparation complete!")
print("Proceed to notebook 02 for optimization model setup.")
print("=" * 60)